# Heritage Short: Wan 2.2 TI2V-5B image-to-video (T4)
Pushed by `pipeline/shots.py` via `pipeline/kaggle_job.py`. The JOBS cell is filled in at push time
(clips to render, keyframe sha256, prompts, seeds). Keyframes arrive as the private dataset
`<user>/heritage-short-wan-in`, named by sha256 so a stale dataset version is detected.
The T4 (15 GB, no bfloat16) cannot hold the 11 GB fp16 text encoder and the 10 GB transformer together, so:
phase 1 encodes every prompt with only the text encoder on the GPU (CPU bf16 if fp16 overflows), then frees it;
phase 2 runs the transformer in float16 with model CPU offload; the VAE stays float32 (tiled).
Writes `out/<shot>.mp4` and `timing.jsonl` (GPU seconds + peak VRAM per attempt) to /kaggle/working.

In [ ]:
# @JOBS (replaced at push time)
JOBS = {"job_id": "local-test", "config": {}, "jobs": []}

In [ ]:
import json, os, sys, time, glob, hashlib, subprocess, traceback
T0 = time.time()
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # less fragmentation on 15 GB
OUT = "/kaggle/working"
os.makedirs(f"{OUT}/out", exist_ok=True)
_log = open(f"{OUT}/timing.jsonl", "a")
def log(**kw):
    """One JSON line per attempt; pipeline/kaggle_job.py copies these into runs.jsonl."""
    _log.write(json.dumps({"job_id": JOBS["job_id"], **kw}) + "\n"); _log.flush(); print(kw)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers==0.39.0", "ftfy", "imageio-ffmpeg"], check=True)

In [ ]:
import gc, numpy as np, torch
from PIL import Image
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
from diffusers.utils import export_to_video

cfg = JOBS["config"]
vae = AutoencoderKLWan.from_pretrained(cfg["model"], subfolder="vae", torch_dtype=torch.float32)
pipe = WanImageToVideoPipeline.from_pretrained(cfg["model"], vae=vae, torch_dtype=torch.float16)
log(id=None, event="setup", seconds=round(time.time() - T0, 1), gpu=torch.cuda.get_device_name(0),
    torch=torch.__version__)

def encode(device, dtype):
    """Phase 1: every prompt through the text encoder while nothing else occupies the GPU."""
    pipe.text_encoder.to(device, dtype)
    out = {}
    with torch.no_grad():
        for job in JOBS["jobs"]:
            pe, ne = pipe.encode_prompt(prompt=job["prompt"], negative_prompt=cfg["negative_prompt"],
                                        do_classifier_free_guidance=cfg["guidance"] > 1, max_sequence_length=512,
                                        device=torch.device(device), dtype=dtype)
            out[job["id"]] = (pe.to("cpu", torch.float16), None if ne is None else ne.to("cpu", torch.float16))
    return out

t = time.time()
torch.cuda.reset_peak_memory_stats()
EMB, where = encode("cuda", torch.float16), "gpu-fp16"
if not all(torch.isfinite(p).all() and (n is None or torch.isfinite(n).all()) for p, n in EMB.values()):
    pipe.text_encoder.to("cpu"); torch.cuda.empty_cache()
    EMB, where = encode("cpu", torch.bfloat16), "cpu-bf16"   # fp16 overflow in the T5 encoder
log(id=None, event="text_encoded", seconds=round(time.time() - t, 1), where=where,
    peak_vram_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2))
pipe.text_encoder.to("cpu")
pipe.text_encoder = None            # phase 2 never needs it: prompts are passed as embeddings
gc.collect(); torch.cuda.empty_cache()
pipe.enable_model_cpu_offload()
if hasattr(pipe.vae, "enable_tiling"):
    pipe.vae.enable_tiling()

In [ ]:
def find_image(name, sha):
    for p in glob.glob(f"/kaggle/input/**/{name}", recursive=True):
        if hashlib.sha256(open(p, "rb").read()).hexdigest() == sha:
            return p
    return None

def fit(img, w, h):
    """Centre-crop to the target aspect, then resize (keyframes are 16:9, Wan wants w x h)."""
    a = w / h
    W, H = img.size
    if W / H > a:
        nw = round(H * a); img = img.crop(((W - nw) // 2, 0, (W - nw) // 2 + nw, H))
    else:
        nh = round(W / a); img = img.crop((0, (H - nh) // 2, W, (H - nh) // 2 + nh))
    return img.resize((w, h), Image.LANCZOS)

for job in JOBS["jobs"]:
    path = find_image(job["image"], job["image_sha256"])
    if path is None:
        log(id=job["id"], status="error", attempt=0, gpu_seconds=0, error="keyframe not found (stale dataset version?)")
        continue
    image = fit(Image.open(path).convert("RGB"), cfg["width"], cfg["height"])
    for attempt in range(1, job["max_attempts"] + 1):
        seed = job["seed"] + attempt - 1          # a retry uses the next seed
        torch.cuda.reset_peak_memory_stats()
        t = time.time()
        try:
            pe, ne = EMB[job["id"]]
            frames = pipe(image=image, prompt_embeds=pe.to("cuda"),
                          negative_prompt_embeds=None if ne is None else ne.to("cuda"),
                          height=cfg["height"], width=cfg["width"], num_frames=job["frames"],
                          num_inference_steps=cfg["steps"], guidance_scale=cfg["guidance"],
                          generator=torch.Generator("cpu").manual_seed(seed), output_type="np").frames[0]
            if not np.isfinite(frames).all() or float(frames.mean()) < 0.02:
                raise RuntimeError("NaN or black frames (float16 overflow?)")
            export_to_video(list(frames), f"{OUT}/out/{job['id']}.mp4", fps=cfg["fps"])
            log(id=job["id"], status="ok", attempt=attempt, seed=seed, gpu_seconds=round(time.time() - t, 1),
                frames=len(frames), file=f"out/{job['id']}.mp4",
                peak_vram_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2))
            break
        except Exception as e:
            log(id=job["id"], status="error", attempt=attempt, seed=seed, gpu_seconds=round(time.time() - t, 1),
                error=repr(e)[:400])
            traceback.print_exc()
            torch.cuda.empty_cache()
log(id=None, event="done", seconds=round(time.time() - T0, 1))